# FireSight-IR | Module 3b — False-Alarm Discriminator (Stage D)

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Platform:** Google Colab (CPU is fine — no GPU needed for this module)

---

## What this module is and why it exists

Module 3a is a general 3-class classifier that sees every fire detection. It achieved:
- Wildfire recall: **0.954**
- False-alarm recall: **0.999**
- False-alarm AUC: **1.000**

That sounds near-perfect. But in an operational FireSat system covering Earth every 20 minutes,
even 0.1% false-alarm leakage = thousands of incorrect fire alerts per season reaching first responders.

**Module 3b is a specialised second-stage discriminator** that runs *after* Module 3a on the pixels
where 3a is uncertain — wildfire probability above a lower threshold but below high confidence.
It focuses exclusively on the hardest discrimination task: **wildfire vs false-alarm**.

The motivating case is the FireSat Ontario first-light image: a 2020 burn scar warmed by the sun
produced elevated LWIR temperature (low BTD, moderate BT_I4). Module 3a might assign such a pixel
a 0.60–0.75 wildfire probability. Module 3b catches it.

## Architecture

**Input:** Pixels where Module 3a wildfire probability is in the ambiguous zone (configurable threshold)
  
**Features (25):**
- Module 3a output: wildfire_prob, fa_prob, nf_prob, max_prob, prob_margin (5)
- Physics: BTD, BT_I4, BT_I5, beer_lambert_proxy, bt_i4_anom_norm, btd_anom_norm (6)
- Surface: lc_urban, lc_forest, lc_grassland, dist_industrial_km, dist_urban_km, is_industrial (6)
- Atmospheric: era5_tcwv, era5_pbl, atm_instability, era5_t2m (4)
- Solar/temporal: sol_zen, is_day, doy_sin, doy_cos (4)

**Model:** XGBoost + LightGBM ensemble (stacked via logistic regression meta-learner)  
**Task:** Binary classification — wildfire (1) vs false-alarm (0) only  
**Target:** false-alarm precision > 0.92, wildfire recall > 0.97 on the ambiguous subset

## Pipeline
```
Input pixel
    ↓
[Module 3a PINN]
    ↓ wildfire_prob
    ├─ prob >= 0.90  →  WILDFIRE (high confidence, skip 3b)
    ├─ prob <= 0.10  →  NOT FIRE (high confidence, skip 3b)
    └─ 0.10 < prob < 0.90  →  [Module 3b XGBoost ensemble]
                                    ↓
                              refined wildfire / false-alarm
```

## Output files
| File | Path |
|---|---|
| Stage D ensemble | `models/firesight_stage_d.pkl` |
| Threshold config | `models/stage_d_config.json` |
| Val predictions | `models/stage_d_val_preds.npy` |
| Figures | `figures/03b_*.png` |

---
## Section 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install xgboost lightgbm scikit-learn shap matplotlib numpy pandas torch h5py pyarrow tqdm -q

import os, json, warnings, pickle, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, roc_auc_score, precision_recall_curve,
    confusion_matrix, average_precision_score, roc_curve
)
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
import lightgbm as lgb
import shap
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = device.type == 'cuda'
print(f'Device: {device}  (GPU not required for Module 3b)')
print('All imports OK.')

---
## Section 1 — Configuration

In [ ]:
BASE_DIR     = '/content/drive/MyDrive/firesight-ir'
MODEL_DIR    = f'{BASE_DIR}/models'
FIGURES_DIR  = f'{BASE_DIR}/figures'
CACHE_DIR    = f'{BASE_DIR}/data/cache'
SPLIT_DIR    = f'{BASE_DIR}/data/splits'
V2_DIR       = f'{BASE_DIR}/data/processed/viirs_fp_v2'

# Module 3a checkpoint
BEST_PATH_3A = f'{MODEL_DIR}/firesight_pinn_best.pt'

# Module 3b outputs
STAGE_D_PATH  = f'{MODEL_DIR}/firesight_stage_d.pkl'
CFG_PATH      = f'{MODEL_DIR}/stage_d_config.json'
PREDS_VAL     = f'{MODEL_DIR}/stage_d_val_preds.npy'
PREDS_TEST    = f'{MODEL_DIR}/stage_d_test_preds.npy'

for d in [MODEL_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# Module 3a dimensions
N_ATM=16; N_SRF=20; N_DERIVED=6; N_CLASSES=3; DROPOUT=0.3

# Stage D thresholds
# Pixels with 3a wildfire_prob in [LOW_THRESH, HIGH_THRESH] go through 3b
# Outside this range = high confidence, 3a decision is used directly
LOW_THRESH  = 0.10   # below this → definitely not fire
HIGH_THRESH = 0.90   # above this → definitely wildfire

# Only look at wildfire vs false-alarm in the ambiguous zone
# (non-fire pixels in ambiguous zone are also passed to 3b as context)
FOCUS_LABELS = [1, 2]  # 1=wildfire, 2=false-alarm

# Feature columns for Stage D (from the parquets)
STAGE_D_FEATS = [
    # Physics — most discriminating
    'BTD', 'BT_I4', 'BT_I5', 'beer_lambert_proxy',
    'bt_i4_anom_norm', 'btd_anom_norm',
    # Surface context — false-alarm signal
    'lc_urban', 'lc_forest', 'lc_grassland', 'lc_cropland', 'lc_shrub',
    'dist_industrial_km', 'dist_urban_km', 'is_industrial',
    # Atmospheric
    'era5_tcwv', 'era5_pbl', 'atm_instability', 'era5_t2m',
    # Solar / temporal
    'sol_zen', 'is_day', 'doy_sin', 'doy_cos',
]

SEED = 42
np.random.seed(SEED)

print('Configuration set.')
print(f'  Stage D features: {len(STAGE_D_FEATS)}')
print(f'  Ambiguous zone: wildfire_prob in [{LOW_THRESH}, {HIGH_THRESH}]')
print(f'  High-confidence bypass: prob < {LOW_THRESH} or prob > {HIGH_THRESH}')

---
## Section 2 — Re-declare Module 3a model and load checkpoint

We need 3a's probability outputs as features for 3b.

In [ ]:
import numpy._core.multiarray
torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])

class ResidualBlock(nn.Module):
    def __init__(self,i,o,d=0.2):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(i,o),nn.BatchNorm1d(o),nn.ReLU(),nn.Dropout(d),nn.Linear(o,o),nn.BatchNorm1d(o))
        self.proj=nn.Linear(i,o) if i!=o else nn.Identity()
    def forward(self,x): return F.relu(self.net(x)+self.proj(x))

class CNNBranch(nn.Module):
    def __init__(self,c=4,d=0.2):
        super().__init__()
        self.enc=nn.Sequential(
            nn.Conv2d(c,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),nn.Conv2d(32,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(True),nn.MaxPool2d(2))
        self.gap=nn.AdaptiveAvgPool2d(1); self.drop=nn.Dropout(d)
    def forward(self,x): return self.drop(self.gap(self.enc(x)).flatten(1))

class FireSightPINN(nn.Module):
    def __init__(self,na=16,ns=20,nd=6,nc=3,dr=0.3):
        super().__init__()
        self.cnn=CNNBranch(4,dr)
        self.atm=nn.Sequential(ResidualBlock(na,64),ResidualBlock(64,32))
        self.srf=nn.Sequential(ResidualBlock(ns,64),ResidualBlock(64,32))
        self.der=nn.Sequential(nn.Linear(nd,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(0.1),nn.Linear(32,16),nn.BatchNorm1d(16),nn.ReLU())
        self.fusion=nn.Sequential(nn.Linear(208,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(dr),nn.Linear(128,64),nn.BatchNorm1d(64),nn.ReLU(),nn.Dropout(dr))
        self.cls=nn.Linear(64,nc)
        self.trans=nn.Sequential(nn.Linear(64,16),nn.ReLU(),nn.Linear(16,1),nn.Sigmoid())
    def forward(self,p,a,s,d):
        f=self.fusion(torch.cat([self.cnn(p),self.atm(a),self.srf(s),self.der(d)],dim=1))
        return self.cls(f),self.trans(f)

model_3a = FireSightPINN(N_ATM,N_SRF,N_DERIVED,N_CLASSES,DROPOUT).to(device)
ckpt = torch.load(BEST_PATH_3A, map_location=device, weights_only=False)
model_3a.load_state_dict(ckpt['model_state_dict'])
model_3a.eval()
print(f'✓ Module 3a loaded: epoch {ckpt["epoch"]} | val_loss={ckpt["val_loss"]:.4f}')

---
## Section 3 — Load cache and get Module 3a probability outputs

In [ ]:
CACHE_FILES = {k: f'{CACHE_DIR}/{k}.npy' for k in ['patches','atm','srf','derived','labels','aux']}

class DS(Dataset):
    def __init__(self,cf,idx):
        self.idx=np.sort(idx)
        self.p=np.load(cf['patches'],mmap_mode='r'); self.a=np.load(cf['atm'],mmap_mode='r')
        self.s=np.load(cf['srf'],mmap_mode='r');     self.d=np.load(cf['derived'],mmap_mode='r')
        self.l=np.load(cf['labels'],mmap_mode='r');  self.x=np.load(cf['aux'],mmap_mode='r')
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        n=int(self.idx[i])
        return (torch.from_numpy(self.p[n].copy()),torch.from_numpy(self.a[n].copy()),
                torch.from_numpy(self.s[n].copy()),torch.from_numpy(self.d[n].copy()),
                torch.tensor(int(self.l[n]),dtype=torch.long),torch.from_numpy(self.x[n].copy()),
                torch.tensor(n, dtype=torch.long))

train_idx = np.load(f'{SPLIT_DIR}/train_index.npy')
val_idx   = np.load(f'{SPLIT_DIR}/val_index.npy')
test_idx  = np.load(f'{SPLIT_DIR}/test_index.npy')
print(f'Splits: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}')

@torch.no_grad()
def get_3a_probs(indices, batch_size=2048, desc='Extracting 3a probs'):
    """Run all pixels through Module 3a and collect softmax probabilities + labels."""
    loader = DataLoader(DS(CACHE_FILES, indices), batch_size=batch_size,
                        shuffle=False, num_workers=0)
    all_probs, all_labels, all_arch_idx = [], [], []
    for p,a,s,d,l,_,arch_idx in tqdm(loader, desc=desc):
        with autocast(enabled=HAS_GPU):
            logits,_ = model_3a(p.to(device),a.to(device),s.to(device),d.to(device))
        all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_labels.append(l.numpy())
        all_arch_idx.append(arch_idx.numpy())
    return (np.concatenate(all_probs),
            np.concatenate(all_labels),
            np.concatenate(all_arch_idx))

print('Extracting Module 3a probabilities (may take ~5-10 min for all splits)...')
train_probs_3a, train_labels, train_arch_idx = get_3a_probs(train_idx, desc='Train')
val_probs_3a,   val_labels,   val_arch_idx   = get_3a_probs(val_idx,   desc='Val')
test_probs_3a,  test_labels,  test_arch_idx  = get_3a_probs(test_idx,  desc='Test')

# Quick accuracy check
for name, probs, labels in [
    ('Train', train_probs_3a, train_labels),
    ('Val',   val_probs_3a,   val_labels),
    ('Test',  test_probs_3a,  test_labels),
]:
    acc = (probs.argmax(1) == labels).mean()
    fa_recall = (probs[labels==2].argmax(1)==2).mean() if (labels==2).sum()>0 else 0.
    print(f'  {name:5s}: acc={acc:.4f} | fa_recall={fa_recall:.4f}')

print('\n✓ Module 3a probabilities extracted.')

---
## Section 4 — Build Stage D feature matrix

Load v2 parquets and join physics/surface features to each pixel by archive index.

In [ ]:
print('Loading v2 parquets to build Stage D feature matrix...')
ALL_YEARS = [2018, 2019, 2020, 2021, 2022, 2023]

dfs = []
for year in ALL_YEARS:
    fp = f'{V2_DIR}/viirs_fp_v2_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['year'] = year
        dfs.append(df)
        print(f'  {year}: {len(df):,} pixels')

df_all = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
print(f'Total: {len(df_all):,} pixels')

# Add BTD if not present
if 'BTD' not in df_all.columns and 'BT_I4' in df_all.columns and 'BT_I5' in df_all.columns:
    df_all['BTD'] = df_all['BT_I4'] - df_all['BT_I5']

# The archive index corresponds to the HDF5 row order
# We need to map arch_idx → parquet row
# Use meta from HDF5
print('\nMapping archive indices to parquet rows via lat/lon...')
with h5py.File(f'{BASE_DIR}/data/processed/patches/firesight_patches.h5', 'r') as hf:
    arch_lats = hf['meta/center_lat'][:].astype(np.float32)
    arch_lons = hf['meta/center_lon'][:].astype(np.float32)
    arch_years = hf['meta/year'][:]

print(f'  Archive: {len(arch_lats):,} entries')

# Build lookup: (year, rounded lat, rounded lon) → parquet row index
# Round to 4 decimal places to handle float32 precision
df_all['lat_r'] = df_all['latitude'].astype(np.float32).round(3)
df_all['lon_r'] = df_all['longitude'].astype(np.float32).round(3)
df_all['yr_int'] = df_all['year'].astype(int)
lookup = {(row.yr_int, row.lat_r, row.lon_r): i for i, row in df_all.iterrows()}

def build_feature_matrix(arch_indices, probs_3a, labels_true):
    """
    Build Stage D feature matrix for a set of archive indices.
    Returns: X (N, n_features+5), y (N,), valid_mask (N,)
    """
    rows = []
    valid = []
    for i, aidx in enumerate(arch_indices):
        yr   = int(arch_years[aidx])
        lat  = round(float(arch_lats[aidx]), 3)
        lon  = round(float(arch_lons[aidx]), 3)
        pidx = lookup.get((yr, lat, lon), None)
        if pidx is None:
            valid.append(False)
            rows.append(None)
            continue
        row = df_all.iloc[pidx]
        feats = {}
        # Stage D raw features
        for col in STAGE_D_FEATS:
            feats[col] = float(row[col]) if col in row.index and pd.notna(row[col]) else 0.0
        # Module 3a probabilities as features
        p = probs_3a[i]
        feats['prob_nf']     = float(p[0])
        feats['prob_wf']     = float(p[1])
        feats['prob_fa']     = float(p[2])
        feats['prob_max']    = float(p.max())
        feats['prob_margin'] = float(sorted(p)[-1] - sorted(p)[-2])
        rows.append(feats)
        valid.append(True)
    valid = np.array(valid)
    valid_rows = [r for r, v in zip(rows, valid) if v]
    X = pd.DataFrame(valid_rows).fillna(0).values.astype(np.float32)
    y = labels_true[valid]
    return X, y, valid

print('Building feature matrices (may take a few minutes)...')
t0 = time.time()
X_train, y_train, valid_train = build_feature_matrix(train_arch_idx, train_probs_3a, train_labels)
X_val,   y_val,   valid_val   = build_feature_matrix(val_arch_idx,   val_probs_3a,   val_labels)
X_test,  y_test,  valid_test  = build_feature_matrix(test_arch_idx,  test_probs_3a,  test_labels)
print(f'Built in {time.time()-t0:.0f}s')
print(f'  Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'  Feature columns: {len(STAGE_D_FEATS)+5}')

FEATURE_NAMES = STAGE_D_FEATS + ['prob_nf','prob_wf','prob_fa','prob_max','prob_margin']
print(f'  Feature names: {FEATURE_NAMES}')

---
## Section 5 — Filter to ambiguous zone

Stage D only processes pixels where Module 3a is uncertain.
High-confidence decisions are passed through unchanged.

In [ ]:
def get_ambiguous_mask(X, y):
    """
    Returns mask of pixels in the ambiguous zone AND are wildfire or false-alarm.
    (Non-fire pixels in ambiguous zone are not the focus of Stage D.)
    """
    prob_wf_col = FEATURE_NAMES.index('prob_wf')
    wf_prob = X[:, prob_wf_col]
    in_zone = (wf_prob > LOW_THRESH) & (wf_prob < HIGH_THRESH)
    is_wf_or_fa = (y == 1) | (y == 2)
    return in_zone & is_wf_or_fa

mask_train = get_ambiguous_mask(X_train, y_train)
mask_val   = get_ambiguous_mask(X_val,   y_val)
mask_test  = get_ambiguous_mask(X_test,  y_test)

X_amb_train = X_train[mask_train]; y_amb_train = (y_train[mask_train] == 1).astype(int)
X_amb_val   = X_val[mask_val];     y_amb_val   = (y_val[mask_val]   == 1).astype(int)
X_amb_test  = X_test[mask_test];   y_amb_test  = (y_test[mask_test] == 1).astype(int)

print('Ambiguous zone analysis:')
for name, X, y, mask in [
    ('Train', X_train, y_train, mask_train),
    ('Val',   X_val,   y_val,   mask_val),
    ('Test',  X_test,  y_test,  mask_test),
]:
    n_total = len(X)
    n_amb = mask.sum()
    n_wf  = (y[mask] == 1).sum()
    n_fa  = (y[mask] == 2).sum()
    print(f'  {name:5s}: {n_total:>9,} total → {n_amb:>7,} ambiguous ({100*n_amb/n_total:.1f}%) | wf={n_wf:,} fa={n_fa:,}')

print(f'\nBinary labels (1=wildfire, 0=false-alarm):')
for name, y_bin in [('Train',y_amb_train),('Val',y_amb_val),('Test',y_amb_test)]:
    print(f'  {name:5s}: wf={y_bin.sum():,} ({100*y_bin.mean():.1f}%) fa={(1-y_bin).sum():,} ({100*(1-y_bin).mean():.1f}%)')

---
## Section 6 — Train Stage D ensemble

XGBoost + LightGBM → stacked via Logistic Regression meta-learner.  
Cross-validation ensures the meta-learner is trained on out-of-fold predictions.

In [ ]:
# ── Base model configs ────────────────────────────────────────────────────────
xgb_params = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, gamma=0.1,
    scale_pos_weight=float((y_amb_train==0).sum()) / float((y_amb_train==1).sum()),
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, n_jobs=-1, verbosity=0,
)
lgb_params = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20,
    class_weight='balanced',
    random_state=SEED, n_jobs=-1, verbose=-1,
)

# ── 5-fold out-of-fold predictions for meta-learner ───────────────────────────
print('Training Stage D ensemble with 5-fold cross-validation...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_xgb = np.zeros(len(y_amb_train))
oof_lgb = np.zeros(len(y_amb_train))

xgb_models, lgb_models = [], []

for fold, (tr_idx, oof_idx) in enumerate(skf.split(X_amb_train, y_amb_train)):
    print(f'  Fold {fold+1}/5...', end=' ')
    Xtr, Xoof = X_amb_train[tr_idx], X_amb_train[oof_idx]
    ytr, yoof = y_amb_train[tr_idx], y_amb_train[oof_idx]

    # XGBoost
    xm = xgb.XGBClassifier(**xgb_params)
    xm.fit(Xtr, ytr, eval_set=[(Xoof, yoof)], early_stopping_rounds=30, verbose=False)
    oof_xgb[oof_idx] = xm.predict_proba(Xoof)[:, 1]
    xgb_models.append(xm)

    # LightGBM
    lm = lgb.LGBMClassifier(**lgb_params)
    lm.fit(Xtr, ytr,
           eval_set=[(Xoof, yoof)],
           callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[oof_idx] = lm.predict_proba(Xoof)[:, 1]
    lgb_models.append(lm)

    auc_x = roc_auc_score(yoof, oof_xgb[oof_idx])
    auc_l = roc_auc_score(yoof, oof_lgb[oof_idx])
    print(f'XGB AUC={auc_x:.4f}  LGB AUC={auc_l:.4f}')

print(f'\nOOF AUC — XGBoost: {roc_auc_score(y_amb_train, oof_xgb):.4f}')
print(f'OOF AUC — LightGBM: {roc_auc_score(y_amb_train, oof_lgb):.4f}')

# ── Meta-learner ──────────────────────────────────────────────────────────────
print('\nFitting meta-learner (Logistic Regression on OOF predictions)...')
meta_X_train = np.column_stack([oof_xgb, oof_lgb])
meta_lr = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
meta_lr.fit(meta_X_train, y_amb_train)
meta_weights = meta_lr.coef_[0]
print(f'  Meta weights: XGB={meta_weights[0]:.3f}  LGB={meta_weights[1]:.3f}')

# ── Final models on full training set ────────────────────────────────────────
print('\nFitting final base models on full training set...')
xgb_final = xgb.XGBClassifier(**xgb_params)
xgb_final.fit(X_amb_train, y_amb_train, verbose=False)
lgb_final = lgb.LGBMClassifier(**lgb_params)
lgb_final.fit(X_amb_train, y_amb_train, callbacks=[lgb.log_evaluation(-1)])

ensemble = {'xgb': xgb_final, 'lgb': lgb_final, 'meta': meta_lr,
            'feature_names': FEATURE_NAMES, 'stage_d_feats': STAGE_D_FEATS}
with open(STAGE_D_PATH, 'wb') as f:
    pickle.dump(ensemble, f)
print(f'✓ Stage D ensemble saved → {STAGE_D_PATH}')

---
## Section 7 — Evaluate Stage D on ambiguous pixels

In [ ]:
def stage_d_predict(X, threshold=0.5):
    """Predict using the ensemble. Returns (proba_wildfire, binary_pred)."""
    p_xgb = xgb_final.predict_proba(X)[:, 1]
    p_lgb = lgb_final.predict_proba(X)[:, 1]
    meta_X = np.column_stack([p_xgb, p_lgb])
    p_meta = meta_lr.predict_proba(meta_X)[:, 1]
    return p_meta, (p_meta >= threshold).astype(int)


print('Evaluating Stage D on ambiguous pixels...')
for name, X_amb, y_amb in [
    ('Val (2023)',         X_amb_val,  y_amb_val),
    ('Test (2018-2022)',   X_amb_test, y_amb_test),
]:
    if len(X_amb) == 0:
        print(f'  {name}: no ambiguous pixels')
        continue
    p, pred = stage_d_predict(X_amb)
    auc = roc_auc_score(y_amb, p)
    ap  = average_precision_score(y_amb, p)
    print(f'\n  Stage D — {name}')
    print(f'  AUC={auc:.4f}  AP={ap:.4f}')
    print(classification_report(y_amb, pred,
          target_names=['false-alarm','wildfire'], digits=4))

# Save val predictions for combined system analysis
p_val, _ = stage_d_predict(X_amb_val)
np.save(PREDS_VAL, np.column_stack([p_val, y_amb_val]))
p_test, _ = stage_d_predict(X_amb_test)
np.save(PREDS_TEST, np.column_stack([p_test, y_amb_test]))
print(f'\n✓ Val predictions saved → {PREDS_VAL}')

---
## Section 8 — Optimal threshold selection

Find the threshold that maximises the F-beta score where beta=2  
(recall weighted 2× more than precision — we prefer not missing real fires).

In [ ]:
p_val_wf, _ = stage_d_predict(X_amb_val)

prec, rec, thresholds = precision_recall_curve(y_amb_val, p_val_wf)
beta = 2.0
fbeta = (1 + beta**2) * (prec * rec) / (beta**2 * prec + rec + 1e-8)
best_thresh_idx = np.argmax(fbeta[:-1])
best_thresh = float(thresholds[best_thresh_idx])
best_fbeta  = float(fbeta[best_thresh_idx])

_, pred_opt = stage_d_predict(X_amb_val, threshold=best_thresh)
print(f'Optimal threshold (F2 score): {best_thresh:.4f}')
print(f'F2 score at threshold: {best_fbeta:.4f}')
print(classification_report(y_amb_val, pred_opt,
      target_names=['false-alarm','wildfire'], digits=4))

cfg = {
    'low_thresh_3a':    LOW_THRESH,
    'high_thresh_3a':   HIGH_THRESH,
    'stage_d_threshold': best_thresh,
    'stage_d_f2':        best_fbeta,
    'feature_names':     FEATURE_NAMES,
    'n_features':        len(FEATURE_NAMES),
}
with open(CFG_PATH, 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'\n✓ Config saved → {CFG_PATH}')

---
## Section 9 — Combined system evaluation (3a + 3b)

Evaluate the full pipeline: 3a handles high-confidence, 3b handles ambiguous zone.

In [ ]:
def combined_predict(X_full, y_full, probs_3a_full, valid_mask, ambiguous_mask):
    """
    Full two-stage pipeline prediction.
    - High-confidence (prob > HIGH or prob < LOW): use 3a directly
    - Ambiguous zone: use Stage D
    """
    prob_wf_3a = probs_3a_full[valid_mask, 1]
    pred_3a    = probs_3a_full[valid_mask].argmax(1)
    final_pred = pred_3a.copy()
    final_prob = probs_3a_full[valid_mask, 1].copy()

    # Override ambiguous wildfire/fa with Stage D
    X_amb = X_full[ambiguous_mask]
    if len(X_amb) > 0:
        p_d, pred_d = stage_d_predict(X_amb, threshold=best_thresh)
        # Map binary pred back: 1=wildfire, 0=false-alarm
        final_pred[ambiguous_mask[valid_mask]] = np.where(pred_d == 1, 1, 2)
        final_prob[ambiguous_mask[valid_mask]] = p_d

    y_true = y_full[valid_mask]
    return final_pred, final_prob, y_true


print('═'*55)
print('  COMBINED SYSTEM (3a + 3b) — Val set 2023')
print('═'*55)

# Rebuild full valid arrays for val set
pred_comb, prob_comb, y_true_comb = combined_predict(
    X_val, y_val, val_probs_3a[valid_val], valid_val, mask_val)

print(classification_report(y_true_comb, pred_comb,
      target_names=['non-fire','wildfire','false-alarm'], digits=4))

print('\n3a only (for comparison):')
print(classification_report(y_val[valid_val], val_probs_3a[valid_val].argmax(1),
      target_names=['non-fire','wildfire','false-alarm'], digits=4))

---
## Section 10 — SHAP feature importance

In [ ]:
BG,PANEL='#0D1117','#161B22'
TEXT,SUBTEXT,BORDER='#E6EDF3','#8B949E','#30363D'
ORANGE,NAVY='#E85D04','#1565C0'
plt.rcParams.update({'figure.facecolor':BG,'axes.facecolor':PANEL,
                     'axes.edgecolor':BORDER,'text.color':TEXT,
                     'xtick.color':SUBTEXT,'ytick.color':SUBTEXT,
                     'axes.labelcolor':SUBTEXT,'grid.color':BORDER})

print('Computing SHAP values for XGBoost (may take ~2 min)...')
sample_size = min(5000, len(X_amb_val))
rng = np.random.default_rng(SEED)
shap_idx = rng.choice(len(X_amb_val), sample_size, replace=False)
X_shap = X_amb_val[shap_idx]

explainer = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X_shap)
mean_abs_shap = np.abs(shap_values).mean(0)
feat_order = np.argsort(mean_abs_shap)[::-1][:20]

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
fig.patch.set_facecolor(BG)

# Panel 1: Feature importance
ax = axes[0]
ax.set_facecolor(PANEL)
feat_names_top = [FEATURE_NAMES[i] for i in feat_order]
feat_vals_top  = mean_abs_shap[feat_order]
colors = [ORANGE if 'prob' in n else NAVY if 'BTD' in n or 'BT_' in n else '#3A5C8A'
          for n in feat_names_top]
ax.barh(range(len(feat_order)), feat_vals_top[::-1], color=colors[::-1], alpha=0.85)
ax.set_yticks(range(len(feat_order)))
ax.set_yticklabels(feat_names_top[::-1], fontsize=9)
ax.set_xlabel('Mean |SHAP value|', fontsize=9)
ax.set_title('Stage D Feature Importance (SHAP)', color=TEXT, fontsize=11)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
handles = [mpatches.Patch(color=ORANGE,label='3a probability features'),
           mpatches.Patch(color=NAVY,label='BTD/BT physics'),
           mpatches.Patch(color='#3A5C8A',label='Other')]
ax.legend(handles=handles, facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)

# Panel 2: PR curve
ax2 = axes[1]
ax2.set_facecolor(PANEL)
p_v, _ = stage_d_predict(X_amb_val)
prec_c, rec_c, _ = precision_recall_curve(y_amb_val, p_v)
ap_c = average_precision_score(y_amb_val, p_v)
ax2.plot(rec_c, prec_c, color=ORANGE, linewidth=2.5, label=f'Stage D  AP={ap_c:.4f}')
ax2.axhline(y_amb_val.mean(), color=BORDER, linewidth=1, linestyle='--', label='Baseline')
ax2.axvline(rec_c[best_thresh_idx], color='white', linewidth=1, linestyle=':', alpha=0.6, label=f'Optimal threshold={best_thresh:.2f}')
ax2.set_xlabel('Recall (wildfire)', fontsize=10)
ax2.set_ylabel('Precision (wildfire)', fontsize=10)
ax2.set_title('Stage D Precision-Recall Curve\n(ambiguous zone only)', color=TEXT, fontsize=11)
ax2.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
ax2.grid(alpha=0.2)
ax2.spines[['top','right']].set_visible(False)
ax2.set_xlim(0,1); ax2.set_ylim(0,1.02)

fig.suptitle('FireSight-IR  |  Module 3b — Stage D False-Alarm Discriminator',
             color=TEXT, fontsize=13, y=1.01)
plt.tight_layout()
out = f'{FIGURES_DIR}/03b_stage_d_analysis.png'
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Section 11 — BTD analysis of remaining errors

In [ ]:
from scipy import stats

if len(X_amb_val) > 0:
    p_val_d, pred_val_d = stage_d_predict(X_amb_val, threshold=best_thresh)
    btd_col = FEATURE_NAMES.index('BTD') if 'BTD' in FEATURE_NAMES else None

    if btd_col is not None:
        btd_vals = X_amb_val[:, btd_col]
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG)
        fig.patch.set_facecolor(BG)

        # Panel 1: BTD by true class in ambiguous zone
        ax = axes[0]
        ax.set_facecolor(PANEL)
        for cls, name, color in [(1,'Wildfire',ORANGE),(0,'False-alarm',NAVY)]:
            sub = btd_vals[y_amb_val == cls]
            if len(sub) < 10: continue
            xr = np.linspace(sub.min(), sub.max(), 300)
            kde = stats.gaussian_kde(sub, bw_method=0.3)
            d = kde(xr); d = d/d.max()
            ax.fill_between(xr, d, alpha=0.25, color=color)
            ax.plot(xr, d, color=color, linewidth=2, label=f'{name} (n={len(sub):,})')
            ax.axvline(np.median(sub), color=color, linewidth=1, linestyle='--', alpha=0.7)
        ax.axvline(10, color='white', linewidth=1.5, linestyle=':', alpha=0.6, label='BTD=10K threshold')
        ax.set_title('BTD of ambiguous pixels\n(what Stage D is discriminating)', color=TEXT, fontsize=10)
        ax.set_xlabel('BTD (K)', fontsize=9)
        ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
        ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)

        # Panel 2: BTD of remaining errors after Stage D
        ax2 = axes[1]
        ax2.set_facecolor(PANEL)
        # False negatives: true wildfire predicted as false-alarm
        fn_mask = (y_amb_val == 1) & (pred_val_d == 0)
        # False positives: true false-alarm predicted as wildfire
        fp_mask = (y_amb_val == 0) & (pred_val_d == 1)
        if fn_mask.sum() > 0:
            ax2.hist(btd_vals[fn_mask], bins=30, alpha=0.6, color=ORANGE,
                     density=True, label=f'Missed wildfire (n={fn_mask.sum():,})')
        if fp_mask.sum() > 0:
            ax2.hist(btd_vals[fp_mask], bins=30, alpha=0.6, color=NAVY,
                     density=True, label=f'False-alarm leak (n={fp_mask.sum():,})')
        ax2.axvline(10, color='white', linewidth=1.5, linestyle=':', alpha=0.6)
        ax2.set_title('BTD of remaining Stage D errors\n(characterising hard cases)', color=TEXT, fontsize=10)
        ax2.set_xlabel('BTD (K)', fontsize=9)
        ax2.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
        ax2.grid(alpha=0.2); ax2.spines[['top','right']].set_visible(False)

        fig.suptitle('FireSight-IR  |  Module 3b — Error Analysis (Val 2023)',
                     color=TEXT, fontsize=12, y=1.01)
        plt.tight_layout()
        out2 = f'{FIGURES_DIR}/03b_error_analysis.png'
        plt.savefig(out2, dpi=160, bbox_inches='tight', facecolor=BG)
        plt.show()
        print(f'Saved → {out2}')

---
## Section 12 — Summary

In [ ]:
print('='*60)
print('  FireSight-IR  |  Module 3b Complete')
print('='*60)
print(f'  Stage D models : XGBoost + LightGBM + LogReg meta')
print(f'  Ambiguous zone : 3a wildfire_prob in [{LOW_THRESH}, {HIGH_THRESH}]')
print(f'  Optimal thresh : {best_thresh:.4f}  (F2 = {best_fbeta:.4f})')
print()
for path, desc in [
    (STAGE_D_PATH, 'Stage D ensemble (pkl)'),
    (CFG_PATH,     'Threshold config (json)'),
    (PREDS_VAL,    'Val predictions (npy)'),
    (f'{FIGURES_DIR}/03b_stage_d_analysis.png', 'SHAP + PR curve figure'),
    (f'{FIGURES_DIR}/03b_error_analysis.png',   'Error analysis figure'),
]:
    e = '✓' if os.path.exists(path) else '✗'
    print(f'  {e}  {desc}')
print()
print('  Next: Module 3c — Uncertainty Quantification')
print('  Or  : Module 4  — Streamlit dashboard')
print('='*60)